# ML-05 — Feature Vector and Leakage/Privacy Check

**Lane:** Freestyle — Growth / Recovery / Momentum Prediction (FlyRank ML Internship)  
**Owner:** Michael Adesiyan  
**Assignment:** Build a multi-signal engineered feature vector, document data availability timelines, and attack the model with a systematic 4-part leakage and privacy audit.

## 1. Build the feature vector

We ingest the 30-day feature window (**March 2026**) and engineer clean, non-leaky signals:
1. **Volume signals:** 30-day sums of Search Console impressions, clicks, GA4 sessions, and word count.
2. **Velocity & Momentum signals:** Second-half vs First-half ratios (Days 16–31 vs Days 1–15) to detect trend slope.
3. **Rank & Quality signals:** Organic CTR, 30-day average Google rank, and daily rank volatility (standard deviation).
4. **Missingness flags:** Explicit boolean flags (`has_search_data`, `has_ga4_data`, `has_word_count`) to avoid category bias when filling missing values.

In [1]:
# Setup Environment, Ingest Data & Engineer Feature Vector
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import roc_auc_score, precision_score

# 1. Authenticate with Hugging Face if token available
load_dotenv(os.path.expanduser("~/.env"))
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)

# 2. Load Feature Month (March 2026), Target Month (April 2026), and Content Dimension
try:
    url_m3 = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
    url_m4 = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet"
    dim_content_url = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"
    
    df_m3 = pd.read_parquet(url_m3)
    df_m4 = pd.read_parquet(url_m4)
    dim_content = pd.read_parquet(dim_content_url)
    print("Loaded live warehouse partition tables from Hugging Face.")
except Exception as e:
    print(f"HF direct read notice: {e}. Falling back to starter CSV dataset.")
    csv_path = "data/raw/content_refresh_anonymized.csv" if os.path.exists("data/raw/content_refresh_anonymized.csv") else "../../data/raw/content_refresh_anonymized.csv"
    df_starter = pd.read_csv(csv_path)
    df_m3 = df_starter.rename(columns={"client_id": "client_hash_id", "content_id": "content_hash_id"}).copy()
    df_m3["report_date"] = pd.to_datetime("2026-03-15")
    df_m3["gsc_impressions"] = df_starter["impressions_90d"] / 3
    df_m3["gsc_clicks"] = df_starter["clicks_90d"] / 3
    df_m3["gsc_avg_position"] = df_starter["avg_position"]
    df_m3["ga4_sessions"] = df_starter["sessions_90d"] / 3
    df_m3["ga4_data_available"] = True
    
    df_m4 = df_starter.rename(columns={"client_id": "client_hash_id", "content_id": "content_hash_id"}).copy()
    df_m4["report_date"] = pd.to_datetime("2026-04-15")
    df_m4["gsc_impressions"] = df_starter["impressions_90d"] / 3 * np.where(df_starter["trend_direction"] == "down", 0.7, 1.1)
    df_m4["ga4_sessions"] = df_starter["sessions_90d"] / 3 * np.where(df_starter["trend_direction"] == "down", 0.7, 1.1)
    df_m4["ga4_data_available"] = True
    dim_content = df_starter.rename(columns={"client_id": "client_hash_id", "content_id": "content_hash_id"})[["content_hash_id", "word_count"]].drop_duplicates()

client_col = "client_hash_id" if "client_hash_id" in df_m3.columns else "client_id"
content_col = "content_hash_id" if "content_hash_id" in df_m3.columns else "content_id"
df_m3["report_date"] = pd.to_datetime(df_m3["report_date"])
df_m4["report_date"] = pd.to_datetime(df_m4["report_date"])

# 3. Engineer Feature Vector (March 2026 Window)
df_m3["is_second_half"] = df_m3["report_date"].dt.day >= 16

# Aggregate base metrics
agg_total = df_m3.groupby([client_col, content_col]).agg(
    gsc_impressions_30d=("gsc_impressions", "sum"),
    gsc_clicks_30d=("gsc_clicks", "sum"),
    gsc_avg_position_30d=("gsc_avg_position", "mean"),
    gsc_position_volatility=("gsc_avg_position", "std"),
    ga4_sessions_30d=("ga4_sessions", "sum"),
    ga4_data_available=("ga4_data_available", "first")
).reset_index()

# Aggregate velocity halves
agg_h1 = df_m3[~df_m3["is_second_half"]].groupby([client_col, content_col]).agg(
    gsc_imp_h1=("gsc_impressions", "sum"),
    gsc_clicks_h1=("gsc_clicks", "sum"),
    ga4_sessions_h1=("ga4_sessions", "sum")
).reset_index()

agg_h2 = df_m3[df_m3["is_second_half"]].groupby([client_col, content_col]).agg(
    gsc_imp_h2=("gsc_impressions", "sum"),
    gsc_clicks_h2=("gsc_clicks", "sum"),
    ga4_sessions_h2=("ga4_sessions", "sum")
).reset_index()

# Merge features
features_df = agg_total.merge(agg_h1, on=[client_col, content_col], how="left").merge(agg_h2, on=[client_col, content_col], how="left")
dim_key = "content_hash_id" if "content_hash_id" in dim_content.columns else "content_id"
features_df = features_df.merge(dim_content[[dim_key, "word_count"]].rename(columns={dim_key: content_col}), on=content_col, how="left")

# Compute momentum ratios
features_df["impression_momentum_ratio"] = (features_df["gsc_imp_h2"].fillna(0) + 1.0) / (features_df["gsc_imp_h1"].fillna(0) + 1.0)
features_df["click_momentum_ratio"] = (features_df["gsc_clicks_h2"].fillna(0) + 1.0) / (features_df["gsc_clicks_h1"].fillna(0) + 1.0)
features_df["session_momentum_ratio"] = (features_df["ga4_sessions_h2"].fillna(0) + 1.0) / (features_df["ga4_sessions_h1"].fillna(0) + 1.0)
features_df["gsc_ctr_30d"] = features_df["gsc_clicks_30d"] / (features_df["gsc_impressions_30d"] + 1.0)

# Compute missingness & availability flags
features_df["has_search_data"] = (features_df["gsc_impressions_30d"] > 0).astype(int)
features_df["has_ga4_data"] = (features_df["ga4_data_available"] == True).astype(int)
features_df["has_word_count"] = (features_df["word_count"].fillna(0) > 0).astype(int)

# Fill missing numerical values cleanly
features_df["gsc_avg_position_30d"] = features_df["gsc_avg_position_30d"].fillna(0)
features_df["gsc_position_volatility"] = features_df["gsc_position_volatility"].fillna(0)
features_df["word_count"] = features_df["word_count"].fillna(0)

FEATURE_COLS = [
    "gsc_impressions_30d", "gsc_clicks_30d", "gsc_avg_position_30d", "gsc_position_volatility", "gsc_ctr_30d",
    "impression_momentum_ratio", "click_momentum_ratio", "session_momentum_ratio",
    "ga4_sessions_30d", "word_count",
    "has_search_data", "has_ga4_data", "has_word_count"
]

# 4. Construct Future Target Label (April 2026 Window)
target_agg = df_m4.groupby([client_col, content_col]).agg(april_impressions=("gsc_impressions", "sum")).reset_index()
dataset = features_df.merge(target_agg, on=[client_col, content_col], how="inner")
dataset["target_decline"] = (dataset["april_impressions"] < 0.85 * dataset["gsc_impressions_30d"]).astype(int)

print(f"=== FEATURE VECTOR CONSTRUCTION COMPLETE ===")
print(f"Total Content Items: {len(dataset):,}")
print(f"Total Engineered Features: {len(FEATURE_COLS)}")
print(f"Target Base Rate (Decline %): {dataset['target_decline'].mean()*100:.2f}%")
print("\nSample Feature Vector (First 3 Rows):")
print(dataset[[client_col, content_col] + FEATURE_COLS[:6]].head(3).to_string(index=False))


Loaded live warehouse partition tables from Hugging Face.
=== FEATURE VECTOR CONSTRUCTION COMPLETE ===
Total Content Items: 331,436
Total Engineered Features: 13
Target Base Rate (Decline %): 29.92%

Sample Feature Vector (First 3 Rows):
         client_hash_id          content_hash_id  gsc_impressions_30d  gsc_clicks_30d  gsc_avg_position_30d  gsc_position_volatility  gsc_ctr_30d  impression_momentum_ratio
client_0797ff3a1fc9a6a5 content_004e9c4c32e88631                    0               0                   0.0                      0.0          0.0                        1.0
client_0797ff3a1fc9a6a5 content_0236ef736698e17c                    0               0                   0.0                      0.0          0.0                        1.0
client_0797ff3a1fc9a6a5 content_025f6cfd3c298870                    0               0                   0.0                      0.0          0.0                        1.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature Name | Category | Meaning & Domain Significance | Missing Value Handling | Available When? |
|---|---|---|---|---|
| `gsc_impressions_30d` | Volume | Total Search Console impressions logged during March 2026. | Defaults to `0.0`. | $\le 2026-03-31$ |
| `gsc_clicks_30d` | Volume | Total organic Search Console clicks in March 2026. | Defaults to `0.0`. | $\le 2026-03-31$ |
| `gsc_avg_position_30d` | Quality | Mean daily Google ranking position in March 2026. | Filled with `0` + flag `has_search_data`. | $\le 2026-03-31$ |
| `gsc_position_volatility` | Volatility | Standard deviation of daily Google rank. Measures rank stability vs turbulence. | Filled with `0.0` (for single-day/no-rank rows). | $\le 2026-03-31$ |
| `gsc_ctr_30d` | Quality | Organic click-through rate: $\frac{\text{clicks}}{\text{impressions} + 1}$. | Explicitly calculated; non-zero denominator. | $\le 2026-03-31$ |
| `impression_momentum_ratio` | Velocity | Ratio of impressions in days 16–31 vs days 1–15. Captures slope of change. | Smoothed ratio with Laplace $+1.0$. | $\le 2026-03-31$ |
| `click_momentum_ratio` | Velocity | Ratio of organic clicks in days 16–31 vs days 1–15. | Smoothed ratio with Laplace $+1.0$. | $\le 2026-03-31$ |
| `session_momentum_ratio` | Velocity | Ratio of GA4 sessions in days 16–31 vs days 1–15. | Smoothed ratio with Laplace $+1.0$. | $\le 2026-03-31$ |
| `ga4_sessions_30d` | Volume | Total GA4 organic user sessions in March 2026. | Defaults to `0.0` + paired with `has_ga4_data`. | $\le 2026-03-31$ |
| `word_count` | Metadata | Content length from `dim_content`. | Imputed with `0` + flagged by `has_word_count`. | Static ($\le 2026-03-31$) |
| `has_search_data` | Flag | Binary indicator ($1$ if impressions > 0, else $0$). | Binary flag (no missingness). | $\le 2026-03-31$ |
| `has_ga4_data` | Flag | Binary indicator ($1$ if `ga4_data_available IS TRUE`, else $0$). | Binary flag from panel metadata. | $\le 2026-03-31$ |
| `has_word_count` | Flag | Binary indicator ($1$ if word count > 0, else $0$). | Binary flag. | $\le 2026-03-31$ |

In [2]:
# Verify Feature Dictionary & Missingness Profile
print("=== FEATURE MISSINGNESS & SUMMARY PROFILE ===")
missing_summary = dataset[FEATURE_COLS].isnull().sum()
print(f"Any remaining NaN values across feature matrix? {'YES (Error!)' if missing_summary.any() else 'NO (Clean!)'}")
print("\nFeature Value Distribution Ranges:")
print(dataset[FEATURE_COLS].describe().T[['mean', 'std', 'min', '50%', 'max']].round(3).to_string())


=== FEATURE MISSINGNESS & SUMMARY PROFILE ===
Any remaining NaN values across feature matrix? NO (Clean!)

Feature Value Distribution Ranges:
                               mean       std    min       50%         max
gsc_impressions_30d         846.793  4044.521  0.000     2.000  617124.000
gsc_clicks_30d                2.480    19.651  0.000     0.000    5668.000
gsc_avg_position_30d          8.532    15.183  0.000     2.412     309.000
gsc_position_volatility       4.633     8.477  0.000     0.000     238.295
gsc_ctr_30d                   0.002     0.017  0.000     0.000       0.667
impression_momentum_ratio    14.427   209.630  0.000     1.000   35481.000
click_momentum_ratio          1.125     1.649  0.009     1.000     463.000
session_momentum_ratio        1.448     3.672  0.017     1.000     714.000
ga4_sessions_30d              3.922    25.382  0.000     0.000    2730.000
word_count                 1618.822  1509.787  0.000  1398.000   29341.000
has_search_data               0.5

## 3. The leakage hunt

We execute a **systematic 4-part attack suite** to test our feature matrix for label leakage and memorization:
1. **Attack 1 (Timeline Boundary Test):** Proves zero temporal overlap ($t_{\text{feature}} \le 2026-03-31$ and $t_{\text{target}} \ge 2026-04-01$).
2. **Attack 2 (Suspicious Correlation Scan):** Scans all features against `target_decline` to confirm no feature has correlation $|r| > 0.85$.
3. **Attack 3 (Ablation / Confession Test):** Injects a known leaky future ratio, proves that the model catches the inflated AUC, and confirms the honest baseline score after purging.
4. **Attack 4 (Grouped vs Random Split Gap):** Evaluates `GroupKFold` (by `client_hash_id`) vs standard `KFold` to measure and report client-memorization bias.

In [3]:
# ATTACK 1: Timeline Boundary Test
max_feature_date = df_m3["report_date"].max()
min_target_date = df_m4["report_date"].min()
timeline_passed = max_feature_date < min_target_date

print("=== ATTACK 1: TIMELINE BOUNDARY TEST ===")
print(f"Latest Feature Date: {max_feature_date.date()} | Earliest Target Date: {min_target_date.date()}")
print(f"Timeline Separation Verdict: {'PASSED (Zero Overlap)' if timeline_passed else 'FAILED (Leakage Detected!)'}")

# ATTACK 2: Suspicious Correlation Scan
print("\n=== ATTACK 2: CORRELATION SCAN (PROXY HUNT) ===")
correlations = dataset[FEATURE_COLS].apply(lambda s: s.corr(dataset["target_decline"]))
max_corr_feature = correlations.abs().idxmax()
max_corr_val = correlations[max_corr_feature]

print(f"Strongest Feature Correlation: '{max_corr_feature}' with r = {max_corr_val:.4f}")
suspicious_features = correlations[correlations.abs() > 0.85]
print(f"Suspicious features with |r| > 0.85: {list(suspicious_features.index)}")
print(f"Correlation Scan Verdict: {'PASSED (No Proxies)' if len(suspicious_features) == 0 else 'FAILED'}")

# ATTACK 3: Ablation & Confession Test
print("\n=== ATTACK 3: ABLATION & CONFESSION TEST ===")
dataset["leaky_future_ratio"] = dataset["april_impressions"] / (dataset["gsc_impressions_30d"] + 1.0)

X_leaky = dataset[FEATURE_COLS + ["leaky_future_ratio"]].fillna(0)
X_honest = dataset[FEATURE_COLS].fillna(0)
y = dataset["target_decline"]

model = LogisticRegression(max_iter=1000)
model.fit(X_leaky, y)
auc_leaky = roc_auc_score(y, model.predict_proba(X_leaky)[:, 1])

model.fit(X_honest, y)
auc_honest = roc_auc_score(y, model.predict_proba(X_honest)[:, 1])

print(f"With Leaky Future Metric (april_impressions ratio): In-Sample ROC-AUC = {auc_leaky:.4f}")
print(f"Honest Engineered Feature Vector: In-Sample ROC-AUC = {auc_honest:.4f}")
dataset.drop(columns=["leaky_future_ratio"], inplace=True)
print("Verdict: Leaky feature confession confirmed and purged.")

# ATTACK 4: Client-Grouped (GroupKFold) vs Random Split Gap
print("\n=== ATTACK 4: GROUPED VS RANDOM SPLIT MEMORIZATION AUDIT ===")
groups = dataset[client_col]
gkf = GroupKFold(n_splits=5)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_cv(splitter, is_grouped=False):
    oof_preds = np.zeros(len(dataset))
    splits = splitter.split(X_honest, y, groups) if is_grouped else splitter.split(X_honest, y)
    for train_idx, val_idx in splits:
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_honest.iloc[train_idx], y.iloc[train_idx])
        oof_preds[val_idx] = clf.predict_proba(X_honest.iloc[val_idx])[:, 1]
    return roc_auc_score(y, oof_preds)

auc_random_cv = evaluate_cv(kf, is_grouped=False)
auc_grouped_cv = evaluate_cv(gkf, is_grouped=True)
memorization_gap = auc_random_cv - auc_grouped_cv

print(f"Random 5-Fold Cross-Validation ROC-AUC: {auc_random_cv:.4f}")
print(f"Client-Grouped 5-Fold Cross-Validation ROC-AUC: {auc_grouped_cv:.4f}")
print(f"Client Memorization Gap: {memorization_gap:+.4f}")
print(f"Analysis: Grouped CV provides the honest estimate for unseen client deployment.")


=== ATTACK 1: TIMELINE BOUNDARY TEST ===
Latest Feature Date: 2026-03-31 | Earliest Target Date: 2026-04-01
Timeline Separation Verdict: PASSED (Zero Overlap)

=== ATTACK 2: CORRELATION SCAN (PROXY HUNT) ===
Strongest Feature Correlation: 'has_search_data' with r = 0.6113
Suspicious features with |r| > 0.85: []
Correlation Scan Verdict: PASSED (No Proxies)

=== ATTACK 3: ABLATION & CONFESSION TEST ===
With Leaky Future Metric (april_impressions ratio): In-Sample ROC-AUC = 0.9967
Honest Engineered Feature Vector: In-Sample ROC-AUC = 0.8637
Verdict: Leaky feature confession confirmed and purged.

=== ATTACK 4: GROUPED VS RANDOM SPLIT MEMORIZATION AUDIT ===
Random 5-Fold Cross-Validation ROC-AUC: 0.8616
Client-Grouped 5-Fold Cross-Validation ROC-AUC: 0.8479
Client Memorization Gap: +0.0137
Analysis: Grouped CV provides the honest estimate for unseen client deployment.


## 4. What I excluded and why

Every excluded field has a strict, documented engineering justification:

1. **`trend_direction` & `trend_pct`:** Excluded because the target label `target_decline` is directly derived from trend percentage changes. Using them would constitute direct label circularity.
2. **`health_score`, `priority_score`, `action_type`:** Excluded because these represent legacy rule-based outputs from existing systems. Learning them produces a circular model that imitates old heuristics rather than discovering underlying traffic patterns.
3. **`april_impressions` & `april_sessions`:** Excluded because these metrics belong strictly to the post-decision outcome window ($t > 2026-03-31$).
4. **`client_hash_id` & `content_hash_id`:** Excluded from feature inputs because raw pseudonym identifiers allow models to memorize specific accounts rather than learning generalizable ranking patterns.

In [4]:
# Programmatic Exclusion Verification
EXCLUSION_REGISTRY = {
    "trend_direction": "Direct label circularity (derived from trend calculation)",
    "trend_pct": "Direct label circularity",
    "health_score": "Legacy product heuristic flag (circular learning)",
    "priority_score": "Legacy product heuristic flag",
    "action_type": "Legacy decision label",
    "april_impressions": "Target-window future outcome metric (knowable only after March 31, 2026)",
    "april_sessions": "Target-window future outcome metric",
    "client_hash_id": "Entity identifier (used for GroupKFold splitting only)",
    "content_hash_id": "Entity identifier (unit of analysis grain key)"
}

print("=== EXCLUSION REGISTRY VERIFICATION ===")
leaked_in_features = [col for col in EXCLUSION_REGISTRY.keys() if col in FEATURE_COLS]
print(f"Any excluded fields found in feature matrix FEATURE_COLS? {'YES (Error!)' if leaked_in_features else 'NO (100% Clean!)'}")
print("\nDocumented Exclusions:")
for col, reason in EXCLUSION_REGISTRY.items():
    print(f"  - [{col}]: {reason}")


=== EXCLUSION REGISTRY VERIFICATION ===
Any excluded fields found in feature matrix FEATURE_COLS? NO (100% Clean!)

Documented Exclusions:
  - [trend_direction]: Direct label circularity (derived from trend calculation)
  - [trend_pct]: Direct label circularity
  - [health_score]: Legacy product heuristic flag (circular learning)
  - [priority_score]: Legacy product heuristic flag
  - [action_type]: Legacy decision label
  - [april_impressions]: Target-window future outcome metric (knowable only after March 31, 2026)
  - [april_sessions]: Target-window future outcome metric
  - [client_hash_id]: Entity identifier (used for GroupKFold splitting only)
  - [content_hash_id]: Entity identifier (unit of analysis grain key)


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.